# Tách nhạc thành nhiều stem bằng BS-RoFormer

Notebook này tách một bản phối thành các file âm thanh riêng. Mặc định dùng **BS-RoFormer-SW**, model 6 stem được toolkit khuyên dùng: vocal, trống, bass, guitar, piano và other (các âm thanh còn lại).

Nếu cần nhiều loại nguồn riêng hơn, có thể đổi sang model Mega 53 stem ở ô cấu hình. Model đó cần GPU có khoảng 16 GB VRAM trở lên; tài liệu toolkit lưu ý các stem riêng lẻ có thể kém hơn model 6 stem.

> Cần bật Internet để cài toolkit và tải model lần đầu. Nên dùng GPU trên Kaggle/Colab. Đặt audio tại `input/input_music.mp3` hoặc `input/input_music.wav`; có thể sửa đường dẫn ở ô cấu hình. File WAV được dùng trực tiếp, không chuyển đổi lại.

## 1. Cấu hình

Đổi `SOURCE_AUDIO` nếu file audio của bạn nằm ở đường dẫn khác. `DEVICE = "auto"` sẽ tự dùng CUDA khi có GPU, nếu không sẽ chạy CPU chậm hơn.

In [ ]:
from pathlib import Path

AUDIO_CANDIDATES = [Path("input/input_music.wav"), Path("input/input_music.mp3")]
SOURCE_AUDIO = next((path for path in AUDIO_CANDIDATES if path.is_file()), AUDIO_CANDIDATES[0])
MODEL_ID = "roformer-model-bs-roformer-sw-by-jarredou"
# Tùy chọn nhiều stem hơn (cần khoảng 16 GB VRAM):
# MODEL_ID = "roformer-model-bs-roformer-mvsep-mega-53-stems"
DEVICE = "auto"

MODEL_DIR = Path("models").resolve()
WAV_INPUT_DIR = Path("input_wav").resolve()
OUTPUT_DIR = Path("output/instruments").resolve()

MODEL_DIR.mkdir(parents=True, exist_ok=True)
WAV_INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Audio nguồn:", SOURCE_AUDIO.resolve())
print("Model:", MODEL_ID)
print("Thư mục kết quả:", OUTPUT_DIR)


## 2. Cài FFmpeg

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg

## 3. Cài BS-RoFormer-Infer

Cài từ repository chính thức vì các bản phát hành trên PyPI hiện là snapshot cũ; bản mới được duy trì và cài trực tiếp từ GitHub.

In [ ]:
!pip install -q "bs-roformer-infer @ git+https://github.com/openmirlab/bs-roformer-infer.git"

## 4. Tải model

Model mặc định khoảng 700 MB. Lần tải đầu cần Internet; các lần sau có thể dùng lại file model đã lưu trong thư mục `models`.

In [ ]:
import subprocess

subprocess.run(
    ["bs-roformer-download", "--list-models"],
    check=True,
)

subprocess.run(
    [
        "bs-roformer-download",
        "--model", MODEL_ID,
        "--output-dir", str(MODEL_DIR),
    ],
    check=True,
)

Nếu file nguồn đã là WAV, notebook sẽ dùng trực tiếp và bỏ qua FFmpeg. Các định dạng khác sẽ được chuyển thành WAV stereo 44.1 kHz.


In [ ]:
if not SOURCE_AUDIO.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy {SOURCE_AUDIO}. "
        "Hãy chép/upload file audio vào thư mục input hoặc sửa SOURCE_AUDIO."
    )

if SOURCE_AUDIO.suffix.lower() == ".wav":
    inference_input_dir = SOURCE_AUDIO.resolve().parent
    prepared_audio = SOURCE_AUDIO.resolve()
    print("File đã là WAV; bỏ qua chuyển đổi:", prepared_audio)
else:
    WAV_INPUT_DIR.mkdir(parents=True, exist_ok=True)
    prepared_audio = WAV_INPUT_DIR / f"{SOURCE_AUDIO.stem}.wav"
    subprocess.run(
        [
            "ffmpeg", "-y",
            "-i", str(SOURCE_AUDIO),
            "-ar", "44100",
            "-ac", "2",
            str(prepared_audio),
        ],
        check=True,
    )
    inference_input_dir = WAV_INPUT_DIR
    print("Đã chuyển đổi:", prepared_audio)


## 6. Tách stem

Kết quả được ghi vào `output/instruments`. Với model mặc định, các stem gồm vocal, drums, bass, guitar, piano và other; toolkit cũng tạo stem instrumental tổng hợp.

In [ ]:
subprocess.run(
    [
        "bs-roformer-infer",
        "--model", MODEL_ID,
        "--models_dir", str(MODEL_DIR),
        "--input_folder", str(inference_input_dir),
        "--store_dir", str(OUTPUT_DIR),
        "--device", DEVICE,
    ],
    check=True,
)


## 7. Xem các file đã tạo

In [ ]:
stem_files = sorted(OUTPUT_DIR.glob("*.wav"))
for stem_file in stem_files:
    print(stem_file)

if not stem_files:
    print("Chưa thấy file WAV trong thư mục kết quả; kiểm tra log inference phía trên.")
